# 4.5 Lab: Support Triage Graph

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

## Your API key - pick ONE option

Some cells in this notebook call a model, and you need your own API key. Run only ONE of the next two cells, whichever fits you:

**Option A - Gemini (free, recommended if you have no OpenAI credit):** Google AI Studio gives a free API key, no card needed. Open https://aistudio.google.com/apikey , create a key, then run the Gemini cell below. The Flash models run on a free tier with rate limits.

**Option B - OpenAI (paid):** create a key at https://platform.openai.com/api-keys and run the OpenAI cell below.

Whichever key you paste is stored only in this notebook session, never saved to the file.

In [ ]:
# Option A: Gemini (free). Run this cell OR the OpenAI one, not both.
import os, getpass
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini (AI Studio) API key (input stays hidden): ")

In [ ]:
# Option B: OpenAI (paid). Run this cell OR the Gemini one, not both.
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Using Gemini instead of OpenAI in the cells below

If you ran the Gemini key cell, one more small step: wherever a cell below creates the model with `ChatOpenAI(...)`, swap that creation line for:

```python
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
```

(Use whatever variable name the cell used, for example `model = ...` instead of `llm = ...`.) Everything else stays the same: prompts, chains, parsers and tools do not care which provider the model came from. For embeddings, swap `OpenAIEmbeddings` for `GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")` from the same package. On the OpenAI path, ignore this note and run the cells exactly as written.

**Prerequisites:** 4.1 through 4.4, and the 0.2 environment. This lab runs with no API key: the classifier is rule-based on purpose, so every run is deterministic and every test is repeatable. An optional LLM step at the end shows where a model would plug in.

## Learning objectives

- Assemble a complete graph from the module's pieces: state with reducers, nodes, conditional routing, streaming, an interrupt, and a checkpointer.
- Put the interrupt exactly where it belongs: a data-dependent approval gate (refunds above a threshold), with the irreversible action after the pause.
- Test a graph the way you test ordinary code: routers and classifiers as unit tests, pause-and-resume as integration tests.
- Read an audit trail from a reducer field and explain what each entry proves about the run.

## What you are building

`triage.py`: a customer-support triage graph. A ticket comes in, gets classified, and routes to one of three handlers: billing, technical, or an urgent escalation. Refunds above 50 EUR pause for human approval; everything else resolves directly. Every step appends to an `audit` log, and the graph emits custom progress events a UI could consume.

The classifier is deliberately rule-based. In production you would use a model here, but a deterministic classifier makes the lab's tests meaningful: when a test fails, the graph is wrong, not the model. The stretch section adds the LLM variant.

## Step 1: the state and the nodes

In [ ]:
import operator
import re
from typing import Annotated

from typing_extensions import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.config import get_stream_writer
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt

REFUND_APPROVAL_THRESHOLD = 50.0

KNOWN_ISSUES = {
    "login": "Reset your password from the login page, then clear your browser cache.",
    "invoice download": "Invoices live under Billing, then History. Use the download icon on the right.",
}


class TicketState(TypedDict):
    ticket_id: str
    subject: str
    body: str
    category: str
    priority: str
    amount: float
    audit: Annotated[list, operator.add]
    draft_reply: str
    approved: bool


def intake(state: TicketState) -> dict:
    get_stream_writer()({"event": "ticket_received", "ticket_id": state["ticket_id"]})
    return {"audit": [f"intake:{state['ticket_id']}"]}


def classify(state: TicketState) -> dict:
    text = (state["subject"] + " " + state["body"]).lower()
    if any(w in text for w in ("outage", "down", "all users")):
        category, priority = "technical", "urgent"
    elif any(w in text for w in ("refund", "charged", "billing", "invoice", "payment")):
        category, priority = "billing", "normal"
    else:
        category, priority = "technical", "normal"
    m = re.search(r"(\d+(?:\.\d{1,2})?)\s*(?:eur|€|\$)", text)
    amount = float(m.group(1)) if m else 0.0
    return {
        "category": category,
        "priority": priority,
        "amount": amount,
        "audit": [f"classified:{category}/{priority}"],
    }


def route(state: TicketState) -> str:
    if state["priority"] == "urgent":
        return "escalate"
    return state["category"]          # "billing" or "technical"


def billing(state: TicketState) -> dict:
    get_stream_writer()({
        "event": "billing_review",
        "ticket_id": state["ticket_id"],
        "amount": state["amount"],
    })
    if state["amount"] > REFUND_APPROVAL_THRESHOLD:
        decision = interrupt({
            "action": "issue_refund",
            "ticket_id": state["ticket_id"],
            "amount": state["amount"],
            "reason": f"Above the {REFUND_APPROVAL_THRESHOLD:.0f} EUR auto-approval limit.",
        })
        approved = decision.get("approved") if isinstance(decision, dict) else bool(decision)
        return {"approved": approved, "audit": [f"refund_review:approved={approved}"]}
    return {
        "draft_reply": ("Thanks for writing. Our billing team handles invoice requests "
                        "within two working days; yours has been logged."),
        "audit": ["billing_faq"],
    }


def route_refund(state: TicketState) -> str:
    return "execute_refund" if state.get("approved") else "finalize"


def execute_refund(state: TicketState) -> dict:
    # The irreversible step lives AFTER the interrupt. The node re-runs on
    # resume, so nothing before the pause may have side effects.
    return {
        "draft_reply": (f"Refund of {state['amount']:.2f} EUR approved and issued "
                        "to your original payment method."),
        "audit": [f"refund_issued:{state['amount']:.2f}"],
    }


def technical(state: TicketState) -> dict:
    text = (state["subject"] + " " + state["body"]).lower()
    for keyword, fix in KNOWN_ISSUES.items():
        if keyword in text:
            return {"draft_reply": f"Known issue. {fix}",
                    "audit": [f"known_issue:{keyword}"]}
    return {"draft_reply": ("Thanks for the report. An engineer has been assigned "
                            "and will follow up."),
            "audit": ["technical_escalated"]}


def escalate(state: TicketState) -> dict:
    get_stream_writer()({
        "event": "page_oncall",
        "ticket_id": state["ticket_id"],
        "priority": "urgent",
    })
    return {"draft_reply": ("We have paged the on-call engineer. "
                            "Expect an update within 30 minutes."),
            "audit": ["escalated:oncall"]}


def finalize(state: TicketState) -> dict:
    updates: dict = {"audit": ["finalized"]}
    if not state.get("draft_reply"):
        updates["draft_reply"] = ("After review, we are unable to issue this refund. "
                                  "A support agent will contact you with alternatives.")
    return updates

Notice where the interrupt sits: inside `billing`, behind a data-dependent condition. Small refunds never pause. The payload is a small readable dict, not the state. And `execute_refund`, the step that actually moves money, is a separate node after the pause, so the re-run rule from lesson 4.4 cannot double-issue anything.

## Step 2: wire the graph

In [ ]:
builder = StateGraph(TicketState)
builder.add_node("intake", intake)
builder.add_node("classify", classify)
builder.add_node("billing", billing)
builder.add_node("execute_refund", execute_refund)
builder.add_node("technical", technical)
builder.add_node("escalate", escalate)
builder.add_node("finalize", finalize)

builder.add_edge(START, "intake")
builder.add_edge("intake", "classify")
builder.add_conditional_edges(
    "classify", route,
    {"billing": "billing", "technical": "technical", "escalate": "escalate"},
)
builder.add_conditional_edges(
    "billing", route_refund,
    {"execute_refund": "execute_refund", "finalize": "finalize"},
)
builder.add_edge("execute_refund", "finalize")
builder.add_edge("technical", "finalize")
builder.add_edge("escalate", "finalize")
builder.add_edge("finalize", END)

graph = builder.compile(checkpointer=InMemorySaver())

Two routers, two different jobs. `route` reads classification results; `route_refund` reads the approval outcome. Both are plain functions of state, which is why both are unit-testable without running the graph.

## Step 3: run the straight-through tickets

In [ ]:
def run_ticket(ticket_id, subject, body):
    config = {"configurable": {"thread_id": ticket_id}}
    result = graph.invoke(
        {"ticket_id": ticket_id, "subject": subject, "body": body, "audit": []},
        config,
    )
    print(ticket_id, "->", result["draft_reply"])
    print("  audit:", result["audit"])
    return result

run_ticket("T-1001", "Invoice copy please",
           "Could you send a copy of my March invoice? No charge issue.")
run_ticket("T-1004", "Cannot log in",
           "I keep getting an error on the login page since this morning.")
run_ticket("T-1005", "Checkout down",
           "Checkout is down for all users, we are losing orders.")

Expected:

In [ ]:
T-1001 -> Thanks for writing. Our billing team handles invoice requests within two working days; yours has been logged.
  audit: ['intake:T-1001', 'classified:billing/normal', 'billing_faq', 'finalized']
T-1004 -> Known issue. Reset your password from the login page, then clear your browser cache.
  audit: ['intake:T-1004', 'classified:technical/normal', 'known_issue:login', 'finalized']
T-1005 -> We have paged the on-call engineer. Expect an update within 30 minutes.
  audit: ['intake:T-1005', 'classified:technical/urgent', 'escalated:oncall', 'finalized']

Read the audit trails: each one is the complete story of the run, in order, because the `operator.add` reducer appended every step's entry. This is the cheapest observability a graph has, and it works with no extra infrastructure.

## Step 4: the approval path, both outcomes

The 120 EUR refund pauses. The 30 EUR refund does not. That contrast is the test:

In [ ]:
# Above the threshold: pauses.
config = {"configurable": {"thread_id": "T-1002"}}
paused = graph.invoke(
    {"ticket_id": "T-1002", "subject": "Refund request",
     "body": "The product broke after a week. I want a refund of 120 EUR.",
     "audit": []},
    config,
)
assert "__interrupt__" in paused
print("paused:", paused["__interrupt__"][0].value)

approved = graph.invoke(Command(resume={"approved": True, "note": "Verified purchase."}), config)
print("reply:", approved["draft_reply"])
print("audit:", approved["audit"])

# Below the threshold: no pause.
small = run_ticket("T-1006", "Refund request",
                   "Charged twice for a 30 EUR plan, please refund one charge.")
assert "__interrupt__" not in small

Expected for the approval:

In [ ]:
paused: {'action': 'issue_refund', 'ticket_id': 'T-1002', 'amount': 120.0, 'reason': 'Above the 50 EUR auto-approval limit.'}
reply: Refund of 120.00 EUR approved and issued to your original payment method.
audit: ['intake:T-1002', 'classified:billing/normal', 'refund_review:approved=True', 'refund_issued:120.00', 'finalized']

Now the rejection, on a fresh thread with the same ticket shape:

In [ ]:
config = {"configurable": {"thread_id": "T-1003"}}
paused = graph.invoke(
    {"ticket_id": "T-1003", "subject": "Refund request",
     "body": "Changed my mind, want a refund of 120 EUR.", "audit": []},
    config,
)
rejected = graph.invoke(Command(resume={"approved": False, "note": "Outside policy."}), config)
print("reply:", rejected["draft_reply"])
print("audit:", rejected["audit"])

In [ ]:
reply: After review, we are unable to issue this refund. A support agent will contact you with alternatives.
audit: ['intake:T-1003', 'classified:billing/normal', 'refund_review:approved=False', 'finalized']

The rejection path is where `finalize` earns its keep: the billing node set no reply on the reject branch, so `finalize` supplies the fallback. A graph with an approval gate needs a defined outcome for every verdict, not just the happy one.

## Step 5: watch it stream

The custom events from `intake`, `billing`, and `escalate` are only visible in `custom` mode. Combine it with `updates` and remember the tuple shape:

In [ ]:
config = {"configurable": {"thread_id": "T-1007"}}
for mode, data in graph.stream(
    {"ticket_id": "T-1007", "subject": "Checkout down",
     "body": "Checkout is down for all users.", "audit": []},
    config,
    stream_mode=["updates", "custom"],
):
    print(mode, "->", data)

In [ ]:
custom -> {'event': 'ticket_received', 'ticket_id': 'T-1007'}
updates -> {'intake': {'audit': ['intake:T-1007']}}
updates -> {'classify': {'category': 'technical', 'priority': 'urgent', 'amount': 0.0, 'audit': ['classified:technical/urgent']}}
custom -> {'event': 'page_oncall', 'ticket_id': 'T-1007', 'priority': 'urgent'}
updates -> {'escalate': {'draft_reply': 'We have paged the on-call engineer. Expect an update within 30 minutes.', 'audit': ['escalated:oncall']}}
updates -> {'finalize': {'audit': ['finalized']}}

This is the shape a real frontend consumes: `custom` events drive the activity feed, `updates` drive the debugger view, and an `__interrupt__` chunk would drive the approval card.

## Step 6: tests

Save these as `test_triage.py`. The first group needs no graph at all:

In [ ]:
from triage import route, route_refund, classify

# Routers are pure functions: test them directly.
assert route({"priority": "urgent", "category": "billing"}) == "escalate"
assert route({"priority": "normal", "category": "billing"}) == "billing"
assert route({"priority": "normal", "category": "technical"}) == "technical"
assert route_refund({"approved": True}) == "execute_refund"
assert route_refund({}) == "finalize"          # rejected or never asked

# The classifier is deterministic: pin its behaviour.
out = classify({"subject": "Refund please", "body": "I was charged 75 EUR twice."})
assert (out["category"], out["priority"], out["amount"]) == ("billing", "normal", 75.0)

The second group is the integration contract: the interrupt fires exactly when it should, and resume works on the same thread:

In [ ]:
from triage import graph
from langgraph.types import Command

def invoke_fresh(ticket_id, body):
    return graph.invoke(
        {"ticket_id": ticket_id, "subject": "Refund request",
         "body": body, "audit": []},
        {"configurable": {"thread_id": ticket_id}},
    )

paused = invoke_fresh("T-9001", "Refund of 120 EUR please.")
assert "__interrupt__" in paused
assert paused["__interrupt__"][0].value["amount"] == 120.0

done = graph.invoke(Command(resume={"approved": True}),
                    {"configurable": {"thread_id": "T-9001"}})
assert "issued" in done["draft_reply"]

straight = invoke_fresh("T-9002", "Refund of 20 EUR please.")
assert "__interrupt__" not in straight

Run with `python test_triage.py`. If the classifier tests pass and the integration tests pass, the lab works; everything else is commentary.

## Lab rubric

| Check | How it is verified |
|---|---|
| Five sample tickets route correctly | `classify` unit tests plus the Step 3 runs |
| Refunds at or below 50 EUR resolve with no pause | `T-1006` / `T-9002`: no `__interrupt__` key |
| Refunds above 50 EUR pause with a readable payload | `T-1002` / `T-9001`: `__interrupt__[0].value` has action, ticket id, amount, reason |
| Approving issues the refund; rejecting sends the fallback reply | the two `Command(resume=...)` runs, audit shows the verdict |
| Audit trail is complete and ordered | every run's `audit` list, first entry `intake:`, last entry `finalized` |
| Resume works on the same thread; a new thread starts fresh | integration tests above |
| Stream consumer handles combined modes | Step 5 prints `(mode, data)` tuples without crashing |

## Stretch goals

1. **Swap the classifier for a model.** Add a `ChatOpenAI` call in `classify` returning a JSON object with category, priority, and amount. Keep the rule-based version behind an environment flag so the tests still run without a key.
2. **Durable checkpoints.** Replace `InMemorySaver` with `SqliteSaver`, pause a refund, kill the process, start a new one, and resume on the same thread id. This is the moment the lab becomes production-shaped.
3. **A second approval tier.** Refunds above 500 EUR need a manager verdict: a second `interrupt()` with a different payload, and the audit must show both approvals in order.
4. **Cost the run.** Count model calls in state when you add the LLM classifier, and route to a cheaper path after a budget. Lesson 6.3 makes this a first-class guardrail.

## A note on the audit log and GDPR

The audit trail is evidence, and evidence is personal data the moment it quotes the customer. This lab's audit stores ticket ids and classification labels, not message bodies, which is a deliberate choice: log what you need to reconstruct the decision, not the customer's words. For an EU deployment, decide three things before the first production run: where the checkpoints live (data residency), how long they are kept (retention), and who can read them (access). The checkpointer that makes interrupts possible is also the store your data-protection answers must cover. Lesson 6.5 works through the full picture.

## What can go wrong

- **Invoking without `audit: []`.** The `operator.add` reducer needs a list to append to. Omit it and the first node that returns an audit entry fails. Initialise every reducer field at `invoke`, as lesson 4.2 taught.
- **Reading `state["approved"]` instead of `state.get("approved")`.** The FAQ branch never sets `approved`; a direct read raises `KeyError` on the path that never pauses. The field exists only on the path that needed it.
- **Forgetting the rejection reply.** An approval gate with only an approve path is a broken feature the first time a human says no. `finalize` supplies the fallback; test the reject path, not just the approve path.
- **Resuming with the wrong thread id.** The approval UI must carry the ticket's thread id through to the resume call. A fresh id starts a fresh run, and the paused ticket waits forever.
- **The interrupt payload growing.** It is tempting to pass the whole ticket "for context". The payload is checkpointed and shown to a reviewer; keep it to the decision's inputs. Context lives in state, which the reviewer can open separately.
- **Testing only with the key present.** If you add the LLM classifier from the stretch goals, keep the deterministic path as the default for tests. A test suite that needs an API key and a network is a test suite that stops running.

## Check your understanding

**1. Why is the refund approval an `interrupt()` inside the billing node rather than an `interrupt_before` breakpoint on the billing node?**
The pause is data-dependent: only refunds above 50 EUR need approval, and the amount is known only after `billing` inspects the ticket. `interrupt_before` pauses unconditionally before the node runs, which would pause every billing ticket including invoice-copy requests. Dynamic `interrupt()` fires on the condition; static breakpoints fire on the schedule.

**2. The `billing` node re-runs from the top on resume. Which line of this lab would cause a real incident if it had a side effect, and how does the lab avoid it?**
Any side effect placed before the `interrupt()` call in `billing` would execute twice: once on the initial run, once on resume. The lab avoids it structurally: `billing` only prepares the decision and pauses, and the money-moving step is a separate node, `execute_refund`, that runs only after the resume.

**3. A rejected refund produces a sensible reply even though the billing node set none. Which node is responsible, and what general rule does this illustrate?**
`finalize`: it fills in a fallback reply when `draft_reply` is empty. The rule is that every branch of an approval gate needs a defined outcome. "The human said no" is a normal path, not an error, and the graph must handle it as deliberately as the approval.

**4. Your audit log shows `refund_review:approved=True` but no `refund_issued` entry, and the run ended at `finalize`. What happened?**
The resume value did not carry approval in the shape the node expected. `route_refund` read `state.get("approved")` as falsy, so the run routed to `finalize` instead of `execute_refund`. Check what the resume call actually passed: the node expects a dict with an `approved` key (or a truthy value), and the caller sent something else.

**5. Why does this lab keep the rule-based classifier as the default even after you add the LLM variant in the stretch goals?**
Determinism. The rule-based classifier makes every test repeatable: a failing test means the graph logic is wrong. An LLM classifier introduces model variance, API cost, and a network dependency into the test suite, so it belongs behind a flag, used for exploration, while the deterministic path guards the behaviour.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.